<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.es/cap05/cap05.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 **Parte Práctica con Ejercicios de Programación**

La presente lista de ejercicios de programación (EP) consolida las formulaciones teóricas presentadas a lo largo del Capítulo 5 — Transformadas y Compresión — mediante una ruta práctica aplicada. Los ejercicios se estructuran a partir de matrices de dimensiones reducidas, lo que permite la validación analítica y la inspección manual de cada coeficiente, manteniendo la consistencia metodológica adoptada en los capítulos anteriores.

El encadenamiento de los ejercicios reproduce rigurosamente el flujo conceptual del capítulo: se comienza con la implementación explícita de la Transformada Discreta de Fourier (DFT) a partir de su definición matemática fundamental; se avanza hacia el diseño de filtros pasa-baja y máscaras *notch* en el dominio de la frecuencia; se aplica la cuantización de coeficientes (núcleo de la compresión con pérdida); y se concluye con la integración de estas etapas en la construcción de un *pipeline* de compresión JPEG simplificado y en el análisis perceptual de formatos de imagen.

> ### ❗ Directrices para la Resolución de los Ejercicios de Programación
>
> En todos los ejercicios de este capítulo, las coordenadas del **centro del espectro** (origen de las frecuencias espaciales tras la aplicación del desplazamiento `fftshift`) deben determinarse mediante división entera. Para una matriz con $L$ filas y $C$ columnas, la componente de frecuencia nula se localiza en la posición:
>
> $$
> (c_y, c_x) = \left( \left\lfloor \frac{L}{2} \right\rfloor, \left\lfloor \frac{C}{2} \right\rfloor \right)
> $$
>
> Esta convención es rigurosamente idéntica a la adoptada por la función `np.fft.fftshift`. Además, en todas las etapas que requieran discretización o redondeo numérico (ya sea en la cuantización de coeficientes AC o en la reconstrucción final de píxeles), debe emplearse el redondeo estándar al entero más cercano (*round half away from zero*), mitigando ambigüedades en valores con fracción exactamente igual a $0.5$.

### 🎯 Objetivo de este Cuaderno

El cuaderno permite desarrollar, validar, organizar y probar soluciones de **Ejercicios de Programación (EPs)** en entornos interactivos, como Colab, con los mismos casos de prueba de Moodle, copiándolos allí solo en el momento de registrar la nota oficial.

#### *Download*

Descargue `morph.py` y `testsuite.py` ejecutando la celda a continuación:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Ejecutando las Pruebas
Para evaluar las pruebas, ejecuta `TestSuite("EP05_01.extensión").run()` en una nueva celda, reemplazando la extensión por la del lenguaje utilizado (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). El sistema descarga los casos de prueba de GitHub, ejecuta el programa y calcula la nota automáticamente.

Para probar código Python directamente, sin guardar archivo, usa `run_code(codigo)` pasando el código como *cadena* en una variable `codigo`:

```python
codigo = """
from morph import mm
# ... tu código aquí ...
"""
TestSuite("EP05_01").run_code(codigo)
```

### EP05_01 🟢 Filtro Pasa-Bajas Ideal por Distancia en el Espectro

En un ***escáner* de documentos antiguo**, el sensor capta papel arrugado y textura de fibra junto con el texto — ruido de alta frecuencia que "contamina" el espectro en los bordes. El técnico de mantenimiento no tiene acceso a la imagen original, solo al **espectro de magnitud ya calculado** por el software del *escáner*. Su trabajo es simple y quirúrgico: mantener únicamente el **círculo central** de bajas frecuencias (la estructura global del documento) y borrar todo lo que esté fuera del radio $D_0$, eliminando la textura fina sin siquiera tocar la imagen espacial.

Este es el **Filtro Pasa-Bajas Ideal (LPFI)**: la operación espectral más directa del capítulo, pero también la que mejor revela la anatomía de un espectro centrado.

#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas) del espectro de magnitud — ya proporcionado **centrado** (equivalente a la salida de `np.fft.fftshift`).
2. **Frecuencia de corte:** Leer el entero $D_0$.
3. **Datos:** Leer los valores enteros de la matriz de magnitud, fila por fila.
4. **Centro del espectro:** Calcular $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
5. **Distancia:** Para cada posición $(u,v)$, calcular
$$
D(u,v) = \sqrt{(u-c_y)^2 + (v-c_x)^2}
$$
6. **Máscara ideal:** Aplicar
$$
H(u,v) = \begin{cases} 1, & D(u,v) \le D_0 \\ 0, & D(u,v) > D_0 \end{cases}
$$
7. **Filtrado:** El valor de salida es $\text{mag}'(u,v) = \text{mag}(u,v) \cdot H(u,v)$.
8. **Salida:** Mostrar la matriz filtrada con dimensiones $L \times C$.

#### 📌 Restricciones Computacionales

* **Comparación no estricta:** el criterio usa $D(u,v) \le D_0$ (la frontera pertenece al filtro, es decir, se mantiene).
* **Tipo:** todos los valores de entrada y salida son enteros; la distancia se calcula en punto flotante solo internamente.
* **Sin redondeo de magnitud:** como la entrada ya es entera y la máscara es binaria (0 o 1), la salida nunca necesita redondeo.

#### 🧠 Fundamentación Teórica

| Región | Distancia al centro | Efecto del filtro |
|---|---|---|
| **Centro** ($D \le D_0$) | Bajas frecuencias | Preservadas — estructura global mantenida |
| **Bordes** ($D > D_0$) | Altas frecuencias | Puestas a cero — textura y ruido eliminados |
| **$D_0$ pequeño** | — | Imagen reconstruida quedaría muy borrosa |
| **$D_0$ grande** | — | Poca filtración; casi toda la energía preservada |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $D_0$.
* Líneas siguientes: Elementos enteros de la matriz de magnitud (centrada).

**Salida:**

* Matriz filtrada en $L$ filas y $C$ columnas, separados por espacio.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 3<br>3<br>1<br>10 20 30<br>40 50 60<br>70 80 90 | 0 20 0<br>40 50 60<br>0 80 0 | Centro $(1,1)$. Las esquinas tienen $D=\sqrt{2}\approx1.41 > 1$, por lo que se ponen a cero; los vecinos ortogonales tienen $D=1 \le 1$ y se mantienen. |
| 1<br>3<br>0<br>5 9 7 | 0 9 0 | $L=1, C=3$: centro en $(0,1)$. Solo la propia posición central ($D=0$) sobrevive a $D_0=0$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0501" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0501 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0501 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0501 button:hover { background: #e8dfcf; }
  #sim-ep0501 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0501_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0501_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0501_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 12px; }
  .sim-ep0501_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim-ep0501_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim-ep0501_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim-ep0501_cell { width: 42px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP05_01: Filtro Pasa-Baja Ideal</span>
  <span class="sim-ep0501_pill">H = (D &le; D₀) ? 1 : 0</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0501_panel" style="margin-bottom:14px;">
    
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Radio de corte (D₀): <span id="sim-ep0501_vl_d0" style="font-family:monospace; color:#26241d;">1</span>
      </label>
    </div>
    
    <input id="sim-ep0501_sl_d0" type="range" min="0" max="4" step="1" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajusta D₀ y observa qué posiciones del espectro 5&times;5 sobreviven al filtro.
    </div>

  </div>

  <!-- Exibição das Grades de Espectro -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0501_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Espectro Original (Magnitud)
      </div>
      <div id="sim-ep0501_grid_orig" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0501_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Resultado Filtrado
      </div>
      <div id="sim-ep0501_grid_new" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0501_debug" class="sim-ep0501_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    –
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep01(root){
    if (!root || root.dataset.sim05Ep01Init) return;
    root.dataset.sim05Ep01Init = "1";

    var N = 5, cy = Math.floor(N / 2), cx = Math.floor(N / 2);
    var mag = [];
    for (var i = 0; i < N; i++){
      var row = [];
      for (var j = 0; j < N; j++){
        row.push(10 * (i + 1) + j + 1);
      }
      mag.push(row);
    }

    var d0el = root.querySelector('#sim-ep0501_sl_d0');
    var d0v  = root.querySelector('#sim-ep0501_vl_d0');
    var go   = root.querySelector('#sim-ep0501_grid_orig');
    var gn   = root.querySelector('#sim-ep0501_grid_new');
    var dbg  = root.querySelector('#sim-ep0501_debug');

    function cellStyle(active){
      if (active) {
        return 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      } else {
        return 'background:#fafaf7; color:#8a8371; border:1px solid #e4dcc8;';
      }
    }

    function render(){
      var D0 = parseInt(d0el.value, 10);
      d0v.textContent = D0;
      go.innerHTML = '';
      gn.innerHTML = '';
      var kept = 0;

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var d = Math.sqrt((i - cy) * (i - cy) + (j - cx) * (j - cx));
          var keep = d <= D0;
          if (keep) kept++;

          var co = document.createElement('div');
          co.className = 'sim-ep0501_cell';
          co.style.cssText = cellStyle(true);
          co.textContent = mag[i][j];
          go.appendChild(co);

          var cn = document.createElement('div');
          cn.className = 'sim-ep0501_cell';
          cn.style.cssText = cellStyle(keep);
          cn.textContent = keep ? mag[i][j] : 0;
          gn.appendChild(cn);
        }
      }

      dbg.textContent = 'Centro = (' + cy + ', ' + cx + ')  |  D₀ = ' + D0 + '  |  Coeficientes mantidos: ' + kept + ' / ' + (N * N);
    }

    d0el.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep01(){
    var root = document.getElementById('sim-ep0501');
    if (root) initSim05Ep01(root); else setTimeout(tryInitSim05Ep01, 200);
  }
  tryInitSim05Ep01();
})();
</script>
""")

**Figura 5.1:** Simulador EP05_01: Filtro Pasabajas Ideal en el Espectro


<figure id="fig-05-sim-ep0501">
  <img src="imagens/fig-05-sim-ep0501.png" alt=" Simulador EP05_01: Filtro Pasabajas Ideal en el Espectro " style="max-width:80%" />
  <figcaption><strong>Figura 5.1:</strong>  Simulador EP05_01: Filtro Pasabajas Ideal en el Espectro </figcaption>
</figure>

In [ ]:
%%writefile EP05_01.py
# Código Python

In [ ]:
TestSuite("EP05_01.py").run()

### EP05_02 🟡 Filtro *Notch*: Eliminando Picos Periódicos

Una cámara de **inspección industrial** captura imágenes de placas de circuito, pero la fuente de alimentación de la línea de producción introduce una **interferencia eléctrica periódica** — un patrón de franjas casi imperceptible a simple vista, pero que aparece en el espectro de Fourier como **pares de picos brillantes** simétricamente posicionados alrededor del centro. El equipo de visión por computadora no puede reprocesar la captura: necesita **localizar y borrar quirúrgicamente** esos pares de picos en el espectro, preservando todo el resto de la información útil de la imagen.

Ese es el papel del **filtro rechaza-banda *notch***: a diferencia del pasa-bajas (que afecta una región continua), ataca **puntos específicos y sus simétricos**, dejando el resto del espectro intacto.

#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas) del espectro de magnitud centrado.
2. **Datos:** Leer los valores enteros de la matriz de magnitud, fila por fila.
3. **Picos:** Leer el entero $K$ (cantidad de pares de picos a eliminar).
4. **Para cada uno de los $K$ picos:** leer tres enteros $\Delta v$, $\Delta u$, $r$ — desplazamiento vertical, desplazamiento horizontal y radio del *notch*.
5. **Centro del espectro:** $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
6. **Supresión simétrica:** para cada pico, poner a cero **todas** las posiciones $(u,v)$ tales que la distancia al punto $(c_y+\Delta v,\, c_x+\Delta u)$ sea $\le r$, **y también** todas las posiciones con distancia $\le r$ al punto simétrico $(c_y-\Delta v,\, c_x-\Delta u)$.
7. **Salida:** Mostrar la matriz resultante con dimensiones $L \times C$.

#### 📌 Restricciones Computacionales

* **Simetría obligatoria:** cada pico informado genera **dos** discos puestos a cero (el punto y su simétrico respecto al centro) — olvidar el simétrico es el error más común.
* **Superposición:** si dos discos se superponen, la posición permanece en cero (no hay "suma" ni restauración).
* **Comparación no estricta:** una posición se pone a cero si $\text{distancia} \le r$.
* **Orden de lectura:** los $K$ picos deben procesarse en el orden en que aparecen en la entrada, pero el resultado final no depende del orden (las operaciones de poner a cero son conmutativas).

#### 🧠 Fundamentación Teórica

| Concepto | Papel en el filtro *notch* |
|---|---|
| **Pico en $(\Delta v, \Delta u)$** | Frecuencia de la interferencia periódica detectada visualmente en el espectro |
| **Punto simétrico $(-\Delta v,-\Delta u)$** | Toda DFT de señal real es hermítica: los picos siempre aparecen en pares simétricos al centro |
| **Radio $r$** | Controla la "anchura" del rechazo — $r$ grande elimina más energía alrededor del pico, pero también información útil |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Líneas siguientes: Elementos enteros de la matriz de magnitud (centrada), $L$ filas.
* Siguiente línea: Entero $K$.
* $K$ líneas siguientes: tres enteros $\Delta v$, $\Delta u$, $r$ (separados por espacios).

**Salida:**

* Matriz resultante en $L$ filas y $C$ columnas, separadas por espacios.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 5<br>5<br>1 2 3 4 5<br>6 7 8 9 10<br>11 12 13 14 15<br>16 17 18 19 20<br>21 22 23 24 25<br>1<br>1 1 0 | 1 2 3 4 5<br>6 0 8 9 10<br>11 12 13 14 15<br>16 17 18 0 20<br>21 22 23 24 25 | Centro $(c_y, c_x) = (2, 2)$. El pico informado $(\Delta v, \Delta u) = (1, 1)$ genera el punto $(3, 3)$ (valor 19) y su simétrico $(1, 1)$ (valor 7), ambos puestos a cero con $r=0$ (solo los puntos exactos). |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0502" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0502 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0502 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0502 button:hover { background: #e8dfcf; }
  #sim-ep0502 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0502_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0502_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0502_grid_ctrls { display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 12px; }
  .sim-ep0502_cell { width: 42px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP05_02: Filtro Notch</span>
  <span class="sim-ep0502_pill">Par Simétrico</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0502_panel" style="margin-bottom:14px;">
    <div class="sim-ep0502_grid_ctrls">
      
      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">&Delta;v</label>
          <span id="sim-ep0502_vl_dv" style="font-family:monospace; font-weight:700; color:#26241d;">1</span>
        </div>
        <input id="sim-ep0502_sl_dv" type="range" min="-2" max="2" step="1" value="1">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">&Delta;u</label>
          <span id="sim-ep0502_vl_du" style="font-family:monospace; font-weight:700; color:#26241d;">1</span>
        </div>
        <input id="sim-ep0502_sl_du" type="range" min="-2" max="2" step="1" value="1">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Radio (r)</label>
          <span id="sim-ep0502_vl_r" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0502_sl_r" type="range" min="0" max="2" step="1" value="0">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:10px; text-align:center;">
      Mueve &Delta;v e &Delta;u para elegir el pico &mdash; observa que el par simétrico también se filtra.
    </div>
  </div>

  <!-- Espectro 5x5 -->
  <div class="sim-ep0502_panel" style="text-align:center; margin-bottom:14px;">
    <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
      Espectro 5&times;5 (Rojo = Eliminado por el Filtro)
    </div>
    <div id="sim-ep0502_grid" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0502_debug" class="sim-ep0502_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep02(root){
    if (!root || root.dataset.sim05Ep02Init) return;
    root.dataset.sim05Ep02Init = "1";

    var N = 5, cy = Math.floor(N / 2), cx = Math.floor(N / 2);
    var mag = [];
    for (var i = 0; i < N; i++){
      var row = [];
      for (var j = 0; j < N; j++){
        row.push(i * 5 + j + 1);
      }
      mag.push(row);
    }

    var dv  = root.querySelector('#sim-ep0502_sl_dv');
    var du  = root.querySelector('#sim-ep0502_sl_du');
    var r   = root.querySelector('#sim-ep0502_sl_r');
    var dvv = root.querySelector('#sim-ep0502_vl_dv');
    var duv = root.querySelector('#sim-ep0502_vl_du');
    var rv  = root.querySelector('#sim-ep0502_vl_r');

    var grid = root.querySelector('#sim-ep0502_grid');
    var dbg  = root.querySelector('#sim-ep0502_debug');

    function cellStyle(kill){
      if (kill) {
        return 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
      } else {
        return 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      }
    }

    function render(){
      var DV = parseInt(dv.value, 10);
      var DU = parseInt(du.value, 10);
      var R  = parseInt(r.value, 10);

      dvv.textContent = DV;
      duv.textContent = DU;
      rv.textContent  = R;

      var p1 = [cy + DV, cx + DU];
      var p2 = [cy - DV, cx - DU];

      grid.innerHTML = '';
      var removed = 0;

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var d1 = Math.sqrt((i - p1[0]) * (i - p1[0]) + (j - p1[1]) * (j - p1[1]));
          var d2 = Math.sqrt((i - p2[0]) * (i - p2[0]) + (j - p2[1]) * (j - p2[1]));
          var kill = (d1 <= R) || (d2 <= R);

          if (kill) removed++;

          var c = document.createElement('div');
          c.className = 'sim-ep0502_cell';
          c.style.cssText = cellStyle(kill);
          c.textContent = kill ? 0 : mag[i][j];
          grid.appendChild(c);
        }
      }

      dbg.textContent = 'Centro = (' + cy + ', ' + cx + ')  |  Pico = (' + p1[0] + ', ' + p1[1] + ')  |  Simétrico = (' + p2[0] + ', ' + p2[1] + ')  |  Removidos: ' + removed;
    }

    [dv, du, r].forEach(function(el){
      el.addEventListener('input', render);
    });

    render();
  }

  function tryInitSim05Ep02(){
    var root = document.getElementById('sim-ep0502');
    if (root) initSim05Ep02(root); else setTimeout(tryInitSim05Ep02, 200);
  }
  tryInitSim05Ep02();
})();
</script>
""")

**Figura 5.2:** Simulador EP05_02: Filtro Notch


<figure id="fig-05-sim-ep0502">
  <img src="imagens/fig-05-sim-ep0502.png" alt=" Simulador EP05_02: Filtro Notch " style="max-width:80%" />
  <figcaption><strong>Figura 5.2:</strong>  Simulador EP05_02: Filtro Notch </figcaption>
</figure>

In [ ]:
%%writefile EP05_02.py
# Código Python

In [ ]:
TestSuite("EP05_02.py").run()

### EP05_03 🟠 Cuantización DCT: la Verdadera Fuente de Compresión

Una aplicación de **galería de fotos** necesita reducir el tamaño de miles de imágenes antes de hacer *upload* a la nube, sin recodificar todo desde cero. El ingeniero responsable ya tiene los **coeficientes DCT** de cada bloque $4\times4$ calculados (la etapa costosa computacionalmente ya se ha realizado) — solo falta aplicar la **tabla de cuantización**, la etapa que realmente descarta información y genera compresión. Los coeficientes de alta frecuencia, menos perceptibles al ojo humano, reciben divisores grandes y tienden a convertirse en **cero**; los coeficientes de baja frecuencia, más perceptibles, reciben divisores pequeños y sobreviven casi intactos.

Vas a implementar exactamente esta etapa: **cuantizar y descuantizar** (dividir, redondear, multiplicar de vuelta) — el corazón de la compresión *lossy* del JPEG.

#### 📋 Directrices de Implementación

1. **Dimensión del bloque:** Leer el entero $N$ (bloque $N \times N$).
2. **Coeficientes:** Leer la matriz $C$ de coeficientes DCT, $N$ filas con $N$ enteros cada una (pueden ser negativos).
3. **Tabla de cuantización:** Leer la matriz $Q$, $N$ filas con $N$ enteros positivos cada una.
4. **Cuantización:** Para cada posición $(u,v)$, calcular el índice cuantizado
$$
\tilde{C}(u,v) = \text{round}\!\left(\frac{C(u,v)}{Q(u,v)}\right)
$$
usando redondeo estándar al entero más cercano (los valores intermedios `.5` nunca ocurren en los casos de prueba).
5. **Descuantización (reconstrucción):** Calcular
$$
C'(u,v) = \tilde{C}(u,v) \times Q(u,v)
$$
6. **Salida:** Mostrar la matriz reconstruida $C'$, $N \times N$, enteros.

#### 📌 Restricciones Computacionales

* ***Round-trip* completo:** la salida es el coeficiente **reconstruido** ($\tilde{C} \times Q$), no el índice cuantizado aislado.
* **División en punto flotante:** la división $C(u,v)/Q(u,v)$ debe realizarse en punto flotante antes del redondeo — la división entera truncada producirá un resultado incorrecto.
* **Signo preservado:** los coeficientes negativos mantienen el signo después de la cuantización y la reconstrucción.
* **$Q(u,v) > 0$ siempre:** no hay necesidad de tratar la división por cero.

#### 🧠 Fundamentación Teórica

| Coeficiente | Frecuencia | Valor típico de $Q$ | Efecto de la cuantización |
|---|---|---|---|
| $C(0,0)$ | DC (promedio del bloque) | Pequeño | Casi siempre sobrevive — domina la energía |
| $C(u,v)$ bajo $u+v$ | Baja frecuencia | Pequeño/medio | Parcialmente preservado |
| $C(u,v)$ alto $u+v$ | Alta frecuencia | Grande | Frecuentemente se convierte en cero — fuente de la compresión |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $N$.
* $N$ líneas siguientes: matriz $C$ (coeficientes DCT, enteros, pueden ser negativos).
* $N$ líneas siguientes: matriz $Q$ (tabla de cuantización, enteros positivos).

**Salida:**

* Matriz reconstruida $C'$, $N \times N$, enteros separados por espacio.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 4<br>50 10 -5 0<br>8 -3 2 1<br>0 1 0 0<br>2 0 0 -1<br>2 5 7 8<br>4 7 8 11<br>6 8 11 12<br>9 11 12 14 | 50 10 -7 0<br>8 0 0 0<br>0 0 0 0<br>0 0 0 0 | $C(0,0)=50/2=25 \to 25\times2=50$ (preservado). $C(0,2)=-5/7\approx-0.71\to-1\to-1\times7=-7$. Ya $C(1,1)=-3/7\approx-0.43\to0$: anulado por la cuantización — la mayor parte del bloque se convierte en cero, ilustrando la compactación de energía en la esquina superior izquierda. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0503" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0503 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0503 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0503 button:hover { background: #e8dfcf; }
  #sim-ep0503 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0503_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0503_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0503_cell { width: 46px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP05_03: Cuantización DCT</span>
  <span class="sim-ep0503_pill">round(C / Q) &times; Q</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0503_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Escala de Q (Agresividad): <span id="sim-ep0503_vl_s" style="font-family:monospace; color:#26241d;">1.00&times;</span>
      </label>
    </div>
    
    <input id="sim-ep0503_sl_s" type="range" min="0.25" max="4" step="0.25" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste la escala de Q y vea cuántos coeficientes sobreviven (no cero) tras el round-trip.
    </div>
  </div>

  <!-- Exibição das Grades 4x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0503_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Coeficientes DCT (C)
      </div>
      <div id="sim-ep0503_grid_c" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0503_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Reconstruido (round(C / Q) &middot; Q)
      </div>
      <div id="sim-ep0503_grid_r" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0503_debug" class="sim-ep0503_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep03(root){
    if (!root || root.dataset.sim05Ep03Init) return;
    root.dataset.sim05Ep03Init = "1";

    var C = [[50, 10, -5, 0], [8, -3, 2, 1], [0, 1, 0, 0], [2, 0, 0, -1]];
    var Qbase = [[2, 5, 7, 8], [4, 7, 8, 11], [6, 8, 11, 12], [9, 11, 12, 14]];

    var s   = root.querySelector('#sim-ep0503_sl_s');
    var sv  = root.querySelector('#sim-ep0503_vl_s');
    var gc  = root.querySelector('#sim-ep0503_grid_c');
    var gr  = root.querySelector('#sim-ep0503_grid_r');
    var dbg = root.querySelector('#sim-ep0503_debug');

    function cell(v, faded){
      var c = document.createElement('div');
      c.className = 'sim-ep0503_cell';
      if (faded) {
        c.style.cssText = 'background:#fafaf7; color:#8a8371; border:1px solid #e4dcc8;';
      } else {
        c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      }
      c.textContent = v;
      return c;
    }

    function render(){
      var scale = parseFloat(s.value);
      sv.innerHTML = scale.toFixed(2) + '&times;';
      gc.innerHTML = '';
      gr.innerHTML = '';
      var zeros = 0, total = 16;

      for (var i = 0; i < 4; i++){
        for (var j = 0; j < 4; j++){
          gc.appendChild(cell(C[i][j], false));
          var Q = Qbase[i][j] * scale;
          var q = Math.round(C[i][j] / Q);
          var rec = Math.round(q * Q);
          if (rec === 0) zeros++;
          gr.appendChild(cell(rec, rec === 0));
        }
      }

      dbg.textContent = 'Ceros: ' + zeros + ' / ' + total + '  |  Quanto maior a escala de Q, mais zeros — maior compressão, menor qualidade.';
    }

    s.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep03(){
    var root = document.getElementById('sim-ep0503');
    if (root) initSim05Ep03(root); else setTimeout(tryInitSim05Ep03, 200);
  }
  tryInitSim05Ep03();
})();
</script>
""")

**Figura 5.3:** Simulador EP05_03: Cuantización DCT (*round-trip*)


<figure id="fig-05-sim-ep0503">
  <img src="imagens/fig-05-sim-ep0503.png" alt=" Simulador EP05_03: Cuantización DCT (*round-trip*) " style="max-width:80%" />
  <figcaption><strong>Figura 5.3:</strong>  Simulador EP05_03: Cuantización DCT (*round-trip*) </figcaption>
</figure>

In [ ]:
%%writefile EP05_03.py
# Código Python

In [ ]:
TestSuite("EP05_03.py").run()

### EP05_04 🔴 Implementando la DFT 2D a partir de la definición

Un laboratorio de investigación en **astronomía computacional** recibió, de una misión antigua, un pequeño sensor experimental cuyos datos brutos no pueden ser procesados por bibliotecas modernas de FFT — el entorno de validación está aislado y solo permite operaciones aritméticas básicas. El equipo necesita **reimplementar la Transformada de Fourier Discreta 2D a partir de la propia definición matemática**, célula por célula, para después comparar bit a bit con `np.fft.fft2` en otro entorno.

Este es el ejercicio más conceptual de la lista: no hay atajos. Vas a implementar el doble sumatorio de la [Equação 5](#eq-05-dft) directamente, evidenciando *por qué* existe la FFT — y el costo computacional que evita.

#### 📋 Directrices de implementación

1. **Dimensiones:** Leer los enteros $M$ (filas) y $N$ (columnas) de la imagen $f(x,y)$.
2. **Datos:** Leer los valores enteros de $f(x,y)$, fila a fila.
3. **DFT 2D:** Para cada par de frecuencias $(u,v)$ con $u=0,\ldots,M-1$ y $v=0,\ldots,N-1$, calcular
$$
F(u,v) = \sum_{x=0}^{M-1}\sum_{y=0}^{N-1} f(x,y)\, e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$
usando la identidad de Euler $e^{-j\theta} = \cos(\theta) - j\sin(\theta)$ para separar parte real e imaginaria — **no se debe utilizar ninguna función de FFT predefinida**.
4. **Magnitud:** Calcular $|F(u,v)| = \sqrt{\text{Re}(F)^2 + \text{Im}(F)^2}$ y redondear al entero más cercano.
5. **Salida:** Mostrar la matriz de magnitudes redondeadas, $M \times N$, en el mismo orden (sin `fftshift` — el DC permanece en $(0,0)$).

#### 📌 Restricciones computacionales

* **Prohibido usar bibliotecas de FFT:** la implementación debe calcular los sumatorios dobles explícitamente (bucles anidados), aunque sea más lenta.
* **Sin `fftshift`:** la salida mantiene la convención cruda de la DFT, con el componente DC en $F(0,0)$ (esquina superior izquierda).
* **Redondeo:** la magnitud final debe redondearse al entero más cercano; en los casos de prueba no hay ambigüedad `.5`.
* **Precisión:** pequeños errores de punto flotante (del orden de $10^{-6}$) antes del redondeo son esperados y no afectan al resultado entero final.

#### 🧠 Fundamentación teórica

| Elemento | Significado |
|---|---|
| $F(0,0)$ | Componente DC — suma de todos los píxeles, $F(0,0) = \sum f(x,y)$ |
| Parte real $\text{Re}(F)$ | Proyección de la señal sobre cosenos |
| Parte imaginaria $\text{Im}(F)$ | Proyección de la señal sobre senos |
| Complejidad de esta implementación | $\mathcal{O}((MN)^2)$ — por eso la FFT, con $\mathcal{O}(MN\log(MN))$, resulta indispensable en imágenes reales |

#### 📦 Especificación de entrada y salida (VPL)

**Entrada:**

* Línea 1: Entero $M$.
* Línea 2: Entero $N$.
* Líneas siguientes: Elementos enteros de $f(x,y)$, $M$ líneas.

**Salida:**

* Matriz de magnitudes $|F(u,v)|$ redondeadas, $M \times N$, separadas por espacios.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 2<br>2<br>1 2<br>3 4 | 10 2<br>4 0 | $F(0,0)=1+2+3+4=10$ (DC = suma total). $F(0,1)=(1-2)+(3-4)=-2 \to |F|=2$. $F(1,0)=(1+2)-(3+4)=-4\to|F|=4$. $F(1,1)=(1-2)-(3-4)=0$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0504" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0504 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0504 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0504 button:hover { background: #e8dfcf; }
  .sim-ep0504_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0504_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0504_cell { width: 52px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 13px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP05_04: DFT 2D &mdash; Definición Directa</span>
  <span class="sim-ep0504_pill">&Sigma;&Sigma; f(x,y) e<sup>-j2&pi;(&hellip;)</sup></span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Instrução -->
  <div class="sim-ep0504_panel" style="margin-bottom:14px; text-align:center;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600;">
      Haz clic en las celdas de f(x,y) para cambiar los valores (incrementa +1; Shift + clic decrementa -1) y observa |F(u,v)| recalculado en vivo.
    </div>
  </div>

  <!-- Exibição das Grades 2x2 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0504_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        f(x,y) &mdash; Dominio Espacial
      </div>
      <div id="sim-ep0504_grid_f" style="display:grid; grid-template-columns:repeat(2, 52px); gap:6px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0504_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        |F(u,v)| &mdash; Magnitud (Sin Shift)
      </div>
      <div id="sim-ep0504_grid_F" style="display:grid; grid-template-columns:repeat(2, 52px); gap:6px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0504_debug" class="sim-ep0504_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep04(root){
    if (!root || root.dataset.sim05Ep04Init) return;
    root.dataset.sim05Ep04Init = "1";

    var f = [[1, 2], [3, 4]];
    var gf  = root.querySelector('#sim-ep0504_grid_f');
    var gF  = root.querySelector('#sim-ep0504_grid_F');
    var dbg = root.querySelector('#sim-ep0504_debug');

    function render(){
      gf.innerHTML = '';
      gF.innerHTML = '';

      for (var x = 0; x < 2; x++){
        for (var y = 0; y < 2; y++){
          (function(xx, yy){
            var c = document.createElement('div');
            c.className = 'sim-ep0504_cell';
            c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7; cursor:pointer;';
            c.textContent = f[xx][yy];
            c.addEventListener('click', function(e){
              if (e.shiftKey){ f[xx][yy]--; } else { f[xx][yy]++; }
              render();
            });
            gf.appendChild(c);
          })(x, y);
        }
      }

      var M = 2, N = 2;
      for (var u = 0; u < M; u++){
        for (var v = 0; v < N; v++){
          var re = 0, im = 0;
          for (var x = 0; x < M; x++){
            for (var y = 0; y < N; y++){
              var theta = 2 * Math.PI * (u * x / M + v * y / N);
              re += f[x][y] * Math.cos(theta);
              im -= f[x][y] * Math.sin(theta);
            }
          }
          var mag = Math.round(Math.sqrt(re * re + im * im));
          var c = document.createElement('div');
          c.className = 'sim-ep0504_cell';
          c.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          c.textContent = mag;
          gF.appendChild(c);
        }
      }

      dbg.textContent = 'F(0,0) = suma de todos los píxeles = ' + (f[0][0] + f[0][1] + f[1][0] + f[1][1]) + ' (componente DC)';
    }

    render();
  }

  function tryInitSim05Ep04(){
    var root = document.getElementById('sim-ep0504');
    if (root) initSim05Ep04(root); else setTimeout(tryInitSim05Ep04, 200);
  }
  tryInitSim05Ep04();
})();
</script>
""")

**Figura 5.4:** Simulador EP05_04: DFT 2D manual


<figure id="fig-05-sim-ep0504">
  <img src="imagens/fig-05-sim-ep0504.png" alt=" Simulador EP05_04: DFT 2D manual " style="max-width:80%" />
  <figcaption><strong>Figura 5.4:</strong>  Simulador EP05_04: DFT 2D manual </figcaption>
</figure>

In [ ]:
%%writefile EP05_04.py
# Código Python

In [ ]:
TestSuite("EP05_04.py").run()

### EP05_05 🏆 *Pipeline* JPEG Completo: DCT, Cuantización y Reconstrucción

Usted ha sido contratado para crear, desde cero, un **códec JPEG didáctico** en un entorno embebido, sin ninguna biblioteca de imágenes disponible — solo operaciones matemáticas básicas. El cliente quiere entender exactamente dónde se pierde la calidad y dónde se recupera, bloque por bloque. Este es el desafío final del capítulo: integrar **todo** lo estudiado — la DCT-II ortonormal, la cuantización perceptual y la reconstrucción vía IDCT — en un único *pipeline* de extremo a extremo, procesando un bloque $N \times N$ desde el inicio hasta el final, exactamente como el estándar JPEG lo hace internamente, $8\times8$ píxeles a la vez.

#### 📋 Directrices de Implementación

1. **Dimensión del bloque:** Leer el entero $N$.
2. **Bloque original:** Leer la matriz de píxeles $f(x,y)$, $N$ líneas con $N$ enteros en $[0,255]$.
3. **Tabla de cuantización:** Leer la matriz $Q$, $N \times N$ enteros positivos.
4. **Centralización:** Restar 128 de cada píxel: $g(x,y) = f(x,y) - 128$.
5. **DCT-II 2D ortonormal:** Calcular
$$
C(u,v) = \alpha(u)\,\alpha(v)\sum_{x=0}^{N-1}\sum_{y=0}^{N-1} g(x,y)\,\cos\!\left[\frac{\pi(2x+1)u}{2N}\right]\cos\!\left[\frac{\pi(2y+1)v}{2N}\right]
$$
con $\alpha(0)=\sqrt{1/N}$ y $\alpha(k)=\sqrt{2/N}$ para $k>0$.
6. **Cuantización:** $\tilde{C}(u,v) = \text{round}(C(u,v)/Q(u,v))$.
7. **Descuantización:** $C'(u,v) = \tilde{C}(u,v)\times Q(u,v)$.
8. **IDCT-II 2D (inversa ortonormal):** Calcular $g'(x,y)$ a partir de $C'(u,v)$ usando la transformada inversa correspondiente (misma base, sumatorio sobre $u,v$).
9. **Reversión de la centralización y redondeo:** $f'(x,y) = \text{round}(g'(x,y) + 128)$, restringido al intervalo $[0,255]$ (*clipping*).
10. **Salida:** Mostrar el bloque reconstruido $f'$, $N \times N$, enteros.

#### 📌 Restricciones Computacionales

* ***Pipeline* completo obligatorio:** todas las seis etapas (centralizar, DCT, cuantizar, descuantizar, IDCT, revertir) deben implementarse — omitir la cuantización no pasa las pruebas, pues el resultado sería idéntico al original.
* ***Clipping*:** los valores reconstruidos fuera de $[0,255]$ deben truncarse (0 si es negativo, 255 si es mayor que 255).
* **Redondeo:** tanto en la cuantización como en la reconstrucción final de los píxeles, use redondeo estándar; los casos de prueba evitan ambigüedad `.5`.
* **Base ortonormal:** la normalización $\alpha(u)$ y $\alpha(v)$ debe aplicarse exactamente como se especifica — sin ella, la IDCT no reconstruye correctamente.

#### 🧠 Fundamentación Teórica

| Etapa | Análoga en el estándar JPEG real | Dónde se pierde la calidad |
|---|---|---|
| Centralización | Misma — la DCT asume señal centrada en cero | Sin pérdida |
| DCT-II | Etapas 3–4 del *pipeline* ([Tabela 5](#tbl-05-pipeline-jpeg)) | Sin pérdida (transformación exacta y reversible) |
| Cuantización | Etapa 5 — división por $Q(u,v)$ | **Principal fuente de pérdida** — los coeficientes de alta frecuencia se vuelven cero |
| IDCT | Reconstrucción final | Reconstruye exactamente los coeficientes *cuantizados*, no los originales |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $N$.
* $N$ líneas siguientes: bloque original $f(x,y)$, enteros en $[0,255]$.
* $N$ líneas siguientes: tabla de cuantización $Q$, enteros positivos.

**Salida:**

* Bloque reconstruido $f'(x,y)$, $N \times N$, enteros en $[0,255]$, separados por espacios.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 4<br>120 130 125 128<br>115 140 135 122<br>118 150 160 130<br>110 120 145 138<br>4 6 8 10<br>6 8 10 12<br>8 10 12 16<br>10 12 16 20 | 118 126 119 131<br>114 143 140 119<br>117 149 159 130<br>107 121 146 139 | Tras la DCT, cuantización agresiva en las altas frecuencias (valores grandes de $Q$ en la esquina inferior derecha) y reconstrucción vía IDCT, el bloque queda **cercano** al original, pero no idéntico — la diferencia es el costo de la compresión *lossy*. |

#### 💡 Consejo de Depuración

Si el resultado no coincide, verifique en este orden: (1) los coeficientes DCT brutos (antes de la cuantización) — deben reconstruir el original **exactamente** vía IDCT si omite las etapas 6–7; (2) la tabla $\alpha(u)$ — error común es aplicar $\sqrt{2/N}$ también para $u=0$; (3) el redondeo de la cuantización, que debe ocurrir **antes** de multiplicar de vuelta por $Q$.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0505" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0505 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0505 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0505 button:hover { background: #e8dfcf; }
  #sim-ep0505 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0505_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0505_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0505_cell { width: 46px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP05_05: Pipeline JPEG (Bloque 4&times;4)</span>
  <span class="sim-ep0505_pill">DCT &rarr; Q &rarr; IDCT</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0505_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Escala de Q (1 = Tabla Base, Mayor = Más Pérdida): <span id="sim-ep0505_vl_s" style="font-family:monospace; color:#26241d;">1.00&times;</span>
      </label>
    </div>
    
    <input id="sim-ep0505_sl_s" type="range" min="0.5" max="5" step="0.5" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajusta el factor de escala de cuantización y observa el bloque reconstruido alejarse (o acercarse) del original.
    </div>
  </div>

  <!-- Exibição das Grades 4x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0505_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Bloque Original
      </div>
      <div id="sim-ep0505_grid_o" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0505_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Reconstruido (DCT &rarr; Q &rarr; IDCT)
      </div>
      <div id="sim-ep0505_grid_r" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0505_debug" class="sim-ep0505_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep05(root){
    if (!root || root.dataset.sim05Ep05Init) return;
    root.dataset.sim05Ep05Init = "1";

    var N = 4;
    var f = [[120, 130, 125, 128], [115, 140, 135, 122], [118, 150, 160, 130], [110, 120, 145, 138]];
    var Qbase = [[4, 6, 8, 10], [6, 8, 10, 12], [8, 10, 12, 16], [10, 12, 16, 20]];

    var s   = root.querySelector('#sim-ep0505_sl_s');
    var sv  = root.querySelector('#sim-ep0505_vl_s');
    var go  = root.querySelector('#sim-ep0505_grid_o');
    var gr  = root.querySelector('#sim-ep0505_grid_r');
    var dbg = root.querySelector('#sim-ep0505_debug');

    function alpha(k){ return k === 0 ? Math.sqrt(1 / N) : Math.sqrt(2 / N); }

    function dct2(g){
      var C = [];
      for (var u = 0; u < N; u++){ C.push(new Array(N).fill(0)); }
      for (var u = 0; u < N; u++){
        for (var v = 0; v < N; v++){
          var sum = 0;
          for (var x = 0; x < N; x++){
            for (var y = 0; y < N; y++){
              sum += g[x][y] * Math.cos(Math.PI * (2 * x + 1) * u / (2 * N)) * Math.cos(Math.PI * (2 * y + 1) * v / (2 * N));
            }
          }
          C[u][v] = alpha(u) * alpha(v) * sum;
        }
      }
      return C;
    }

    function idct2(C){
      var g = [];
      for (var x = 0; x < N; x++){ g.push(new Array(N).fill(0)); }
      for (var x = 0; x < N; x++){
        for (var y = 0; y < N; y++){
          var sum = 0;
          for (var u = 0; u < N; u++){
            for (var v = 0; v < N; v++){
              sum += alpha(u) * alpha(v) * C[u][v] * Math.cos(Math.PI * (2 * x + 1) * u / (2 * N)) * Math.cos(Math.PI * (2 * y + 1) * v / (2 * N));
            }
          }
          g[x][y] = sum;
        }
      }
      return g;
    }

    function cell(v){
      var c = document.createElement('div');
      c.className = 'sim-ep0505_cell';
      c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      c.textContent = v;
      return c;
    }

    function render(){
      var scale = parseFloat(s.value);
      sv.innerHTML = scale.toFixed(2) + '&times;';
      go.innerHTML = '';
      gr.innerHTML = '';

      var g = [];
      for (var x = 0; x < N; x++){
        var row = [];
        for (var y = 0; y < N; y++){
          row.push(f[x][y] - 128);
        }
        g.push(row);
      }

      var C = dct2(g);
      var Cq = [];
      for (var u = 0; u < N; u++){
        var row = [];
        for (var v = 0; v < N; v++){
          var Qv = Qbase[u][v] * scale;
          var q = Math.round(C[u][v] / Qv);
          row.push(q * Qv);
        }
        Cq.push(row);
      }

      var gr2 = idct2(Cq);
      var diffSum = 0, n = 0;

      for (var x = 0; x < N; x++){
        for (var y = 0; y < N; y++){
          go.appendChild(cell(f[x][y]));
          var rec = Math.round(gr2[x][y] + 128);
          rec = Math.max(0, Math.min(255, rec));
          gr.appendChild(cell(rec));
          diffSum += Math.abs(rec - f[x][y]);
          n++;
        }
      }

      dbg.textContent = 'Error medio absoluto por píxel: ' + (diffSum / n).toFixed(2) + '  |  Quanto maior a escala de Q, maior o erro de reconstrução.';
    }

    s.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep05(){
    var root = document.getElementById('sim-ep0505');
    if (root) initSim05Ep05(root); else setTimeout(tryInitSim05Ep05, 200);
  }
  tryInitSim05Ep05();
})();
</script>
""")

**Figura 5.5:** Simulador EP05_05: *Pipeline* JPEG completo en bloques


<figure id="fig-05-sim-ep0505">
  <img src="imagens/fig-05-sim-ep0505.png" alt=" Simulador EP05_05: *Pipeline* JPEG completo en bloques " style="max-width:80%" />
  <figcaption><strong>Figura 5.5:</strong>  Simulador EP05_05: *Pipeline* JPEG completo en bloques </figcaption>
</figure>

In [ ]:
%%writefile EP05_05.py
# Código Python

In [ ]:
TestSuite("EP05_05.py").run()